# 01. Fixed window

**Tier:** Deep. **Why you'd use this:** A fixed window is `k` consecutive values. The next window adds the new value and drops the one that fell off the left.

**After this lesson you can:** maintain a sum (or count) for a window of length k; slide by adding one value and dropping one.
About 20 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. What stays the same when a fixed window of length k slides by one?
2. How do you update the window sum without rescanning k values?

<details><summary>Answers</summary>

For 01 fixed window lesson: skim [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb) if needed.
</details>


## Recognize it

You need the best score over every block of exactly `k` consecutive values. The prompt says "subarray of length k", "every window of size k", or "maximum sum of k neighbors".

Signals: contiguous values, fixed length `k`, and a running aggregate (sum, count, average). A look-alike that is not this pattern: a window whose length grows and shrinks to meet a condition — that is a variable window.

If `k` is missing and the length itself is the answer under a threshold, leave this lesson and open the shortest-window one. Fixed means the size is an input, not something you discover.


## Why it works this way

A fixed window of length `k` is a contiguous slice. The next window shares `k - 1` values with the previous one. Recomputing `sum` from scratch each time rescans those shared values, so the work per window is proportional to `k` and the total work across `n - k + 1` windows grows like `n · k`.

The sliding update uses the overlap. When the window moves one step right, one new value enters on the right and one old value leaves on the left. Add the newcomer, subtract the departure, and the sum is current again in constant time. Across the whole list you pay `O(k)` to open the first window and `O(1)` per slide, so total time stays proportional to `n`. Extra memory is a few integers for the running sum and the best score.

The invariant after each slide: `window` equals the sum of `nums[index - k + 1 : index + 1]`. That is why the departure index is `index - k` when `index` is the new right edge. Off-by-one errors usually subtract the wrong left value or start the loop at the wrong index.

This pattern needs non-negative thinking only for some variable-window problems. Fixed windows work with negatives too: a negative entering simply lowers the sum. Prefer a prefix array when you must answer many arbitrary range queries, not a stream of equal-length windows. Prefer a nested rescan only when `k` and `n` are both tiny and clarity matters more than the linear pass.

The design reason is reuse of overlap. You are not inventing a new aggregate each step; you are correcting yesterday's aggregate by the two boundary edits. That correction is why timed challenges treat fixed windows as the gateway to sliding techniques before variable windows add shrink rules.


## When to use it

Use a fixed window when the length is a constant `k` and you want a max, min, or average over every such block. Use a variable window when the length is part of the answer under a constraint. Use prefix sums when ranges are arbitrary, not sliding by one. Skip the slide when `k == n` and a single sum is enough.

For medians or other order statistics inside the window, a plain sum slide is not enough — you need a multiset or two heaps. Keep that upgrade separate from the basic sum/count case taught here.


## Brute force, and why it's slow

Resum each window of length `k` from scratch. Count how many additions that performs.


In [1]:
def max_window_sum_slow(nums, k):
    additions = 0
    best = None
    for start in range(len(nums) - k + 1):
        total = 0
        for index in range(start, start + k):
            total += nums[index]
            additions += 1
        best = total if best is None else max(best, total)
    return best, additions


for n, k in ((20, 5), (40, 5), (80, 5)):
    print(n, k, max_window_sum_slow(list(range(n)), k))


20 5 (85, 80)
40 5 (185, 180)
80 5 (385, 380)


## Key insight

The next window of length `k` reuses `k - 1` values. Add the entering value and subtract the leaving value instead of scanning `k` cells again. Track the best score as you slide.


## Worked example

Return the largest sum of any `k` consecutive values.


In [2]:
def max_window_sum(nums: list[int], k: int) -> int:
    # 1. Open the first window once.
    window = sum(nums[:k])
    best = window
    # 2. Slide: add the newcomer, drop the value that left.
    for index in range(k, len(nums)):
        window += nums[index] - nums[index - k]
        best = max(best, window)
    return best


print(max_window_sum([2, 1, 5, 1, 3, 2], 3))
print(max_window_sum([2, 3, 4, 1, 5], 2))


9
7


## Trace

`[2, 1, 5, 1, 3, 2]`, `k = 3`:

| right | window values | sum | best |
|---|---|---|---|
| 2 | 2, 1, 5 | 8 | 8 |
| 3 | 1, 5, 1 | 7 | 8 |
| 4 | 5, 1, 3 | 9 | **9** |
| 5 | 1, 3, 2 | 6 | 9 |


## Cost

Opening costs `k` additions. Each of the remaining `n - k` slides costs a constant update. Total time grows with `n`. Extra memory is constant. The rescan version pays about `k` work per window and grows like `n · k`. Negatives do not change that asymptotic picture for fixed length.


## Variants

Not solved here: [Sliding Window Maximum](../../challenges/problems/44-sliding-window-maximum.ipynb), [Maximum Subarray](../../challenges/problems/21-maximum-subarray.ipynb), [Longest Substring Without Repeating Characters](../../challenges/problems/03-longest-substring-without-repeating-characters.ipynb).


## Common mistake

This bug is silent. The update subtracts `nums[index - k + 1]` instead of `nums[index - k]`, so it drops the wrong left value and the max is wrong. There is no traceback. The print is `8`, and the answer is `9`.


In [3]:
def max_window_sum_wrong(nums, k):
    window = sum(nums[:k])
    best = window
    for index in range(k, len(nums)):
        window += nums[index] - nums[index - k + 1]  # wrong departure
        best = max(best, window)
    return best


print(max_window_sum_wrong([2, 1, 5, 1, 3, 2], 3))


8


The fix subtracts `nums[index - k]`, the value that just left the window. The buggy cell prints `8` for `[2, 1, 5, 1, 3, 2]` with `k = 3` because the leaving value stays in the sum.


In [4]:
print(max_window_sum([2, 1, 5, 1, 3, 2], 3))


9


## Python idioms

`sum(nums[:k])` opens the window. A deque of indexes is the usual next step when you need the maximum *value* in each window, not the sum — that variant lives with monotonic deques.


In [5]:
def min_window_sum(nums: list[int], k: int) -> int:
    window = sum(nums[:k])
    best = window
    for index in range(k, len(nums)):
        window += nums[index] - nums[index - k]
        best = min(best, window)
    return best


print(min_window_sum([2, 1, 5, 1, 3, 2], 3))


6


## Recap

- A fixed window keeps length `k`.
- Slide by adding one value and dropping one.
- Track the best aggregate as you go.

Next: [practice](01-fixed-window-practice.ipynb).
